# 05 · Final model

The final model is tuned on **all 109 development boreholes** with the same procedure as the inner CV (groups = the 5 outer folds), then trained on all of them, exactly as V3.

**Which model:** the one chosen by V3's 1-SE rule in notebook 04, **unless** `settings.toml → [final] override_model` is set. An override must have a written reason (`override_reason`); both the rule's choice and the override are recorded and must be reported in the thesis.

The model is saved in **this** project's folder. V3's frozen model is never changed; when the final model is the same as V3's, its predictions are checked against V3's frozen file.

**Run time:** 1-30 minutes (XGB about 1 minute).

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| V3 folder:", S["v3_root"])
import joblib

cells = pd.read_csv(H.processed_dir(S) / "cells.csv")
dev = cells[cells.set == "development"]
sel = json.loads((H.out_dir(S, "04_results") / "selected_model.json").read_text())
RULE, BLOCK = sel["selected"], H.block_sizes(S)[0]
OVERRIDE = S.get("final", {}).get("override_model") or None
REASON = S.get("final", {}).get("override_reason", "").strip()
if OVERRIDE and not REASON:
    raise ValueError("override_model is set but override_reason is empty: write the reason in settings.toml")
NAME = OVERRIDE or RULE
OUT, PRIV = H.out_dir(S, "05_final_model"), H.out_dir(S, "05_final_model", private=True)
print("Chosen by the 1-SE rule:", RULE)
print("FINAL MODEL:", NAME, f"(override; reason: {REASON})" if OVERRIDE and OVERRIDE != RULE else "")
print(f"{BLOCK} m blocks | {dev.BH_Code.nunique()} boreholes, {len(dev)} cells")

In [ ]:
from tqdm import tqdm
bar = tqdm(total=H.n_trials(NAME, S), desc=f"final {NAME}", bar_format="{desc} {percentage:3.0f}%|{bar:25}| {n_fmt}/{total_fmt} trials [{elapsed}<{remaining}]")
final = H.fit_final(NAME, dev, BLOCK, S, on_trial=lambda: bar.update(1))
bar.close()
path = PRIV / f"final_model_A_{NAME}_{BLOCK}m.joblib"
joblib.dump(final["model"], path, compress=3)
rec = {"model": NAME, "selected_by_rule": RULE, "override": bool(OVERRIDE and OVERRIDE != RULE),
       "override_reason": REASON if OVERRIDE and OVERRIDE != RULE else None,
       "block_m": BLOCK, "params": final["params"], "threshold": final["threshold"],
       "cv_macro_f1_all_development": final["inner_macro_f1"], "n_train_boreholes": int(dev.BH_Code.nunique()),
       "n_train_cells": len(dev), "top_elevation_range_m": [float(dev.Top_Elevation_m.min()), float(dev.Top_Elevation_m.max())],
       "model_file": path.name, "model_sha256": H.sha256(path)}
(OUT / "final_model.json").write_text(json.dumps(rec, indent=1, default=str))
final["trials"].to_csv(OUT / "final_model_optuna_trials.csv", index=False)
print(json.dumps(rec, indent=1, default=str))

## Comparison model
For the external tests and the figures, the other model (the rule's choice, or XGB if the rule's choice is final) is shown beside the final model. It is V3's frozen model when that is the rule's choice (read only), otherwise it is trained here the same way.

In [ ]:
v3_rec = json.loads((H.v3_outputs(S, "05_final") / "final_model.json").read_text())
v3_path = H.v3_outputs(S, "05_final", "restricted") / v3_rec["model_file"]
assert H.sha256(v3_path) == v3_rec["model_sha256"], "V3's frozen model file has changed!"
v3_model = H.load_v3_model(S, v3_path)
X = cells[H.FEATURES].values
OTHER = RULE if NAME != RULE else "XGB"
if NAME == v3_rec["model"]:
    same_p = np.array_equal(final["model"].predict_proba(X)[:, 1], v3_model.predict_proba(X)[:, 1])
    print("Final model = V3's final model. Parameters:", final["params"] == v3_rec["params"],
          "| threshold:", final["threshold"] == v3_rec["threshold"], "| predictions IDENTICAL:", same_p)
if OTHER == v3_rec["model"]:
    other = {"model": OTHER, "threshold": v3_rec["threshold"], "params": v3_rec["params"],
             "model_file": v3_path.name, "source": "V3 frozen model (read only)"}
    other_model = v3_model
else:
    o = H.fit_final(OTHER, dev, BLOCK, S)
    other_model = o["model"]
    joblib.dump(other_model, PRIV / f"comparison_model_A_{OTHER}_{BLOCK}m.joblib", compress=3)
    other = {"model": OTHER, "threshold": o["threshold"], "params": o["params"],
             "model_file": f"comparison_model_A_{OTHER}_{BLOCK}m.joblib", "source": "trained here"}
joblib.dump(other_model, PRIV / "comparison_model.joblib", compress=3)
(OUT / "comparison_model.json").write_text(json.dumps(other, indent=1, default=str))
print("Comparison model:", OTHER, "| threshold", other["threshold"], "|", other["source"])
print(f"Finished in {(time.time() - T0) / 60:.1f} min. Next: notebook 06 (or 07 if 06 is already done).")